# EFSM Phase 2 — Dataset Preparation

**Goal:** Download `facebook/empathetic_dialogues`, format as ChatML conversations, and save to `data/train.jsonl`, `data/val.jsonl`, `data/test.jsonl`.

**Expected outcomes:**
- `data/train.jsonl` ~17,000 rows (80% stratified split)
- `data/val.jsonl`   ~2,100 rows  (10%)
- `data/test.jsonl`  ~2,100 rows  (10%)
- Each row is a JSON array of ChatML message dicts with emotion-tagged user turns
- EFSMDataset label mask verified: only assistant tokens unmasked

---
**Before running:** No GPU needed — run on Kaggle CPU. Add `HF_TOKEN` as a Kaggle Secret.

## Cell 1 — Load Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN']      = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

print('Secrets loaded.')

## Cell 2 — Clone Repo and Install Dependencies

> **Note:** We pin `datasets<3.0.0` because v3.x removed support for the legacy loading script used by `facebook/empathetic_dialogues`. The `--upgrade` flag forces Kaggle's pre-installed newer version to be downgraded.

In [ ]:
!git clone https://github.com/tasbidrahman10/empathetic-voice-llm.git
%cd empathetic-voice-llm
# Force-install datasets<3.0.0 FIRST (Kaggle ships datasets 3.x which dropped legacy script support)
!pip install "datasets>=2.18.0,<3.0.0" --upgrade -q
!pip install -r requirements.txt -q

## Cell 3 — Run Preprocessing

Downloads `facebook/empathetic_dialogues`, builds ChatML conversations, and saves the three JSONL splits.

In [ ]:
!python src/data/preprocess_empathetic.py --config configs/config.yaml

## Cell 4 — Spot-Check: Display Sample Conversations

Load `data/train.jsonl` and print 3 formatted conversations to verify emotion tags and ChatML structure.

In [ ]:
import json

TRAIN_PATH = 'data/train.jsonl'

with open(TRAIN_PATH) as f:
    samples = [json.loads(line) for line in f]

print(f'Total train conversations: {len(samples)}')
print()

for idx in [0, 500, 1000]:
    print(f'--- Conversation {idx} ---')
    for msg in samples[idx]:
        role = msg['role'].upper()
        content = msg['content']
        if role == 'SYSTEM':
            content = content[:80] + '...'
        print(f'[{role}] {content}')
    print()

## Cell 5 — Verify EFSMDataset: Tokenisation and Label Masking

Loads the Qwen tokenizer (text-only, no model weights needed), instantiates `EFSMDataset`, and checks that the label mask correctly exposes only assistant-turn tokens.

In [ ]:
import torch
from transformers import AutoTokenizer
from huggingface_hub import login
from src.data.dataset import EFSMDataset

login(token=os.environ['HF_TOKEN'])

MODEL_ID = 'Qwen/Qwen2.5-Omni-7B'
print('Loading tokenizer (no model weights downloaded) ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ['HF_TOKEN'])

dataset = EFSMDataset('data/train.jsonl', tokenizer, max_seq_len=2048)
print(f'Dataset length: {len(dataset)}')

sample = dataset[0]
input_ids = sample['input_ids']
labels    = sample['labels']

unmasked = (labels != -100).sum().item()
total    = labels.shape[0]
masked   = total - unmasked

print(f'\nSample[0]:')
print(f'  input_ids shape : {input_ids.shape}')
print(f'  labels shape    : {labels.shape}')
print(f'  Masked tokens   : {masked} / {total}  ({100*masked/total:.1f}%)')
print(f'  Unmasked tokens : {unmasked} / {total}  ({100*unmasked/total:.1f}%)')
print()

assistant_ids = input_ids[labels != -100]
assistant_text = tokenizer.decode(assistant_ids, skip_special_tokens=True)
print(f'Assistant tokens decoded (first 200 chars):')
print(assistant_text[:200])

## Cell 6 — Upload JSONL Files to HuggingFace Hub

Backs up the three JSONL files to `{HF_USERNAME}/efsm-checkpoints` on HuggingFace so they persist across Kaggle sessions and can be downloaded for Phase 3 training.

In [ ]:
from huggingface_hub import HfApi
import yaml, os

with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

# hf_hub_checkpoint_repo in config is now "tasbid001/efsm-checkpoints"
CHECKPOINT_REPO = cfg['model']['hf_hub_checkpoint_repo']

api = HfApi(token=os.environ['HF_TOKEN'])

for local_path in ['data/train.jsonl', 'data/val.jsonl', 'data/test.jsonl']:
    abs_path = os.path.abspath(local_path)   # absolute path avoids CWD confusion
    if not os.path.isfile(abs_path):
        print(f'ERROR: {abs_path} not found — did Cell 3 run successfully?')
        break
    filename = os.path.basename(local_path)
    api.upload_file(
        path_or_fileobj=abs_path,
        path_in_repo=f'data/{filename}',
        repo_id=CHECKPOINT_REPO,
        repo_type='model',
        commit_message=f'Phase 2: upload {filename}',
    )
    print(f'Uploaded {abs_path}  →  {CHECKPOINT_REPO}/data/{filename}')

print('\nPhase 2 COMPLETE. All JSONL files backed up to HuggingFace Hub.')